# Notebook 04 - Fairness audit and faithfulness refinements

Two jobs. First, the fairness audit proper (C4): whether the at-risk flag fires equitably
across protected groups. Second, two corrections that NB03's results made necessary:
a predicted-class fix to the deletion metric so logistic regression stops reading negative,
and a base-rate adjustment that tests whether the deprived-vs-affluent faithfulness gap
survives once the higher base risk of deprived students is accounted for.

**Part A - fairness audit (C4).** From the saved test-set predictions, per protected group:
flag rate, true-positive rate, and false-positive rate; then the demographic-parity,
equal-opportunity, and FPR gaps with 1,000-resample bootstrap CIs, across deprivation
(deprived 0-30% vs affluent 70-100%), disability, and age band.

**Part B - faithfulness, corrected and adjusted.**
* Predicted-class deletion AOPC: confidence in the predicted class, not the positive class,
  so removing important features always reduces confidence and the sign is interpretable for
  every model.
* Base-rate-adjusted gap: the deprived-minus-affluent deletion gap recomputed within
  predicted-risk bands and pooled, with a bootstrap CI. If the adjusted gap collapses toward
  zero, the raw gap was a base-rate artifact; if it holds, it is a genuine effect.

**Inputs (from NB02):** `predictions_week{w}`, `models_week{w}.joblib`, `model_ready_week{w}`.
**Outputs:** `results/fairness_summary.csv`, `results/faithfulness_adjusted_summary.csv`,
and figures in `results/figures/`.

## 0. Setup

In [1]:
try:
    import shap
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'shap'])
    import shap

from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/StudentEWS_Research/student-ews-research')
except Exception:
    ROOT = Path('.')

PROC   = ROOT / 'results' / 'processed'
MODELS = ROOT / 'results' / 'models'
FIG    = ROOT / 'results' / 'figures'
FIG.mkdir(parents=True, exist_ok=True)

CUTOFF_WEEKS = [5, 10, 15, 25]
SEED   = 42
B_BOOT = 1000
N_SHAP = 1200            # sample for the faithfulness recompute (RF SHAP is the slow part)
STEPS  = [1, 2, 3, 5, 8, 12]
MODEL_ORDER = ['logreg', 'rf', 'hgb']
RISK_BINS = 4            # predicted-risk strata for the base-rate adjustment

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

## 1. Helpers

In [3]:
def load(stem):
    for e in ('.parquet', '.csv'):
        if (Path(str(stem) + e)).exists():
            return pd.read_parquet(str(stem) + e) if e == '.parquet' else pd.read_csv(str(stem) + e)
    return None

def save_table(df, stem):
    try:
        p = Path(str(stem) + '.parquet'); df.to_parquet(p, index=False)
    except Exception:
        p = Path(str(stem) + '.csv'); df.to_csv(p, index=False)
    return p

# ---------- fairness ----------
def _rates(y, pred):
    y, pred = np.asarray(y), np.asarray(pred)
    flag = pred.mean() if len(pred) else np.nan
    tpr = pred[y == 1].mean() if (y == 1).any() else np.nan
    fpr = pred[y == 0].mean() if (y == 0).any() else np.nan
    return flag, tpr, fpr

def fairness_gaps(yA, predA, yB, predB, B, seed):
    yA, predA, yB, predB = map(np.asarray, (yA, predA, yB, predB))
    fA, fB = _rates(yA, predA), _rates(yB, predB)
    raw = np.array(fA) - np.array(fB)                      # dp, eo, fpr
    r = np.random.default_rng(seed)
    nA, nB = len(yA), len(yB)
    boot = np.empty((B, 3))
    for i in range(B):
        ia, ib = r.integers(0, nA, nA), r.integers(0, nB, nB)
        boot[i] = np.array(_rates(yA[ia], predA[ia])) - np.array(_rates(yB[ib], predB[ib]))
    ci = np.nanpercentile(boot, [2.5, 97.5], axis=0)        # (2,3)
    return fA, fB, raw, ci

# ---------- SHAP + predicted-class faithfulness ----------
def shap_matrix(sv):
    if isinstance(sv, list):
        sv = sv[1] if len(sv) == 2 else sv[-1]
    sv = np.asarray(sv)
    if sv.ndim == 3:
        sv = sv[:, :, 1] if sv.shape[-1] == 2 else sv[:, :, -1]
    return sv

def compute_shap(name, est, Xdf):
    if name == 'logreg':
        scaler, lr = est.named_steps['scaler'], est.named_steps['clf']
        bg = scaler.transform(Xdf)
        return shap_matrix(shap.LinearExplainer(lr, bg).shap_values(scaler.transform(Xdf)))
    try:
        return shap_matrix(shap.TreeExplainer(est).shap_values(Xdf, check_additivity=False))
    except Exception:
        return shap_matrix(shap.Explainer(est.predict_proba, Xdf)(Xdf).values)

def make_predict_fn(est, features):
    def f(Xnp):
        return est.predict_proba(pd.DataFrame(Xnp, columns=features))[:, 1]
    return f

def deletion_aopc_predclass(predict_fn, Xnp, abs_shap, steps):
    # AOPC on the confidence of the PREDICTED class (sign-correct for every model).
    n, F = Xnp.shape
    base = Xnp.mean(axis=0)
    order = np.argsort(-abs_shap, axis=1)
    rows = np.arange(n)
    p1 = predict_fn(Xnp)
    cls = (p1 >= 0.5).astype(int)
    conf = lambda pp: np.where(cls == 1, pp, 1 - pp)
    c0 = conf(p1)
    dpi = np.zeros(n)
    for k in steps:
        Xd = Xnp.copy(); cols = order[:, :k]
        for j in range(k):
            Xd[rows, cols[:, j]] = base[cols[:, j]]
        dpi += (c0 - conf(predict_fn(Xd)))
    return dpi / len(steps), p1

# ---------- base-rate adjusted gap ----------
def _pooled_gap(aopc, risk, dmask, amask, nbins):
    sel = dmask | amask
    qs = np.quantile(risk[sel], np.linspace(0, 1, nbins + 1)); qs[0] -= 1e-9; qs[-1] += 1e-9
    binid = np.digitize(risk, qs[1:-1])
    gaps, wts = [], []
    for b in range(nbins):
        d, a = dmask & (binid == b), amask & (binid == b)
        if d.sum() and a.sum():
            gaps.append(aopc[d].mean() - aopc[a].mean()); wts.append(int(d.sum() + a.sum()))
    return np.average(gaps, weights=wts) if gaps else np.nan

def gap_raw_and_adjusted(aopc, risk, dmask, amask, B, seed, nbins):
    raw = aopc[dmask].mean() - aopc[amask].mean()
    adj = _pooled_gap(aopc, risk, dmask, amask, nbins)
    di, ai = np.where(dmask)[0], np.where(amask)[0]
    r = np.random.default_rng(seed)
    boots = np.empty(B)
    for i in range(B):
        idx = np.concatenate([r.choice(di, len(di), True), r.choice(ai, len(ai), True)])
        dm = np.zeros(len(idx), bool); dm[:len(di)] = True
        boots[i] = _pooled_gap(aopc[idx], risk[idx], dm, ~dm, nbins)
    lo, hi = np.nanpercentile(boots, [2.5, 97.5])
    return raw, adj, lo, hi

## 2. Part A - fairness audit (C4)

In [4]:
DEPRIVED, AFFLUENT = [0, 1, 2], [7, 8, 9]

fair_rows = []
for w in CUTOFF_WEEKS:
    pr = load(PROC / f'predictions_week{w}')
    pr['imd_ord'] = pd.to_numeric(pr['imd_band_ord'], errors='coerce')
    for model in MODEL_ORDER:
        pred = pr[f'pred_{model}']; y = pr['at_risk']
        comparisons = {
            'imd':   (pr['imd_ord'].isin(DEPRIVED), pr['imd_ord'].isin(AFFLUENT), 'deprived', 'affluent'),
            'disability': (pr['disability'] == 'Y', pr['disability'] == 'N', 'disabled', 'not_disabled'),
            'age':   (pr['age_band'] == '0-35', pr['age_band'] == '55<=', 'age_0_35', 'age_55plus'),
        }
        for attr, (mA, mB, gA, gB) in comparisons.items():
            if mA.sum() == 0 or mB.sum() == 0:
                continue
            fA, fB, raw, ci = fairness_gaps(y[mA], pred[mA], y[mB], pred[mB], B_BOOT, SEED)
            fair_rows.append({
                'cutoff_week': w, 'model': model, 'attribute': attr,
                'group_A': gA, 'group_B': gB, 'n_A': int(mA.sum()), 'n_B': int(mB.sum()),
                'flag_A': fA[0], 'flag_B': fB[0], 'dp_gap': raw[0], 'dp_lo': ci[0,0], 'dp_hi': ci[1,0],
                'tpr_A': fA[1], 'tpr_B': fB[1], 'eo_gap': raw[1], 'eo_lo': ci[0,1], 'eo_hi': ci[1,1],
                'fpr_A': fA[2], 'fpr_B': fB[2], 'fpr_gap': raw[2], 'fpr_lo': ci[0,2], 'fpr_hi': ci[1,2],
            })
    print(f'week {w:2d}: fairness audit done')

fairness = pd.DataFrame(fair_rows)
fairness.to_csv(ROOT / 'results' / 'fairness_summary.csv', index=False)
print('\nEqual-opportunity gap (TPR_A - TPR_B), with 95% CI:')
print(fairness[['cutoff_week','model','attribute','eo_gap','eo_lo','eo_hi']].round(4).to_string(index=False))

week  5: fairness audit done
week 10: fairness audit done
week 15: fairness audit done
week 25: fairness audit done

Equal-opportunity gap (TPR_A - TPR_B), with 95% CI:
 cutoff_week  model  attribute  eo_gap   eo_lo  eo_hi
           5 logreg        imd  0.0635  0.0255 0.1050
           5 logreg disability  0.0305 -0.0150 0.0730
           5 logreg        age  0.0272 -0.1598 0.2185
           5     rf        imd  0.0515  0.0138 0.0912
           5     rf disability  0.0194 -0.0256 0.0643
           5     rf        age -0.0325 -0.2142 0.1586
           5    hgb        imd  0.0405  0.0011 0.0812
           5    hgb disability  0.0254 -0.0215 0.0688
           5    hgb        age  0.0276 -0.1604 0.2175
          10 logreg        imd  0.0676  0.0314 0.1050
          10 logreg disability  0.0395 -0.0008 0.0809
          10 logreg        age -0.0156 -0.1717 0.1588
          10     rf        imd  0.0376  0.0041 0.0730
          10     rf disability  0.0455  0.0040 0.0851
          10     rf  

## 3. Part B - faithfulness, corrected and base-rate-adjusted

In [ ]:
faith_rows, demo = [], {}
for w in CUTOFF_WEEKS:
    bundle = joblib.load(MODELS / f'models_week{w}.joblib')
    FEATURES = bundle['features']
    ready = load(PROC / f'model_ready_week{w}')
    test = ready[ready['split'] == 'test']
    samp = (test if len(test) <= N_SHAP
            else test.groupby('at_risk', group_keys=False).sample(frac=N_SHAP/len(test), random_state=SEED))
    samp = samp.reset_index(drop=True)
    Xdf = samp[FEATURES].reset_index(drop=True)
    Xnp = Xdf.to_numpy(dtype=float)
    imd = pd.to_numeric(samp['imd_band_ord'], errors='coerce').to_numpy()
    dmask = np.isin(imd, [0, 1, 2]); amask = np.isin(imd, [7, 8, 9])

    for model in MODEL_ORDER:
        est = bundle['models'][model]['estimator']
        M = compute_shap(model, est, Xdf)
        aopc, risk = deletion_aopc_predclass(make_predict_fn(est, FEATURES), Xnp, np.abs(M), STEPS)
        raw, adj, lo, hi = gap_raw_and_adjusted(aopc, risk, dmask, amask, B_BOOT, SEED, RISK_BINS)
        faith_rows.append({
            'cutoff_week': w, 'model': model,
            'del_aopc_predclass': float(aopc.mean()),
            'raw_gap_imd': raw, 'adj_gap_imd': adj, 'adj_lo': lo, 'adj_hi': hi,
        })
        if w == 15 and model == 'hgb':
            demo = {'aopc': aopc, 'risk': risk, 'dmask': dmask, 'amask': amask}
    print(f'week {w:2d}: faithfulness recompute done')

faith = pd.DataFrame(faith_rows)
faith.to_csv(ROOT / 'results' / 'faithfulness_adjusted_summary.csv', index=False)
print('\nDeletion AOPC (predicted-class) and raw vs base-rate-adjusted imd gap:')
print(faith.round(4).to_string(index=False))

## 4. Figures

In [ ]:
hgb_f = fairness[fairness['model'] == 'hgb']

# (a) Equal-opportunity gap by cutoff, per protected attribute (hgb)
plt.figure(figsize=(6.4, 4))
for attr, g in hgb_f.groupby('attribute'):
    g = g.sort_values('cutoff_week')
    plt.errorbar(g['cutoff_week'], g['eo_gap'],
                 yerr=[g['eo_gap']-g['eo_lo'], g['eo_hi']-g['eo_gap']],
                 marker='o', capsize=3, label=attr)
plt.axhline(0, color='#444441', lw=0.8)
plt.xlabel('week cutoff'); plt.ylabel('equal-opportunity gap (TPR_A - TPR_B)')
plt.title('Fairness: equal-opportunity gap by cutoff (gradient boosting)')
plt.legend(frameon=False); plt.tight_layout()
plt.savefig(FIG / 'fig_fairness_eo_gap.png', dpi=200); plt.close()

# (b) Corrected deletion AOPC by model (all should be positive now)
plt.figure(figsize=(6.0, 4))
for model, g in faith.groupby('model'):
    g = g.sort_values('cutoff_week')
    plt.plot(g['cutoff_week'], g['del_aopc_predclass'], marker='o', label=model)
plt.axhline(0, color='#444441', lw=0.8)
plt.xlabel('week cutoff'); plt.ylabel('deletion AOPC (predicted class)')
plt.title('Faithfulness with predicted-class fix (all models positive)')
plt.legend(frameon=False); plt.tight_layout()
plt.savefig(FIG / 'fig_deletion_aopc_corrected.png', dpi=200); plt.close()

# (c) Raw vs base-rate-adjusted imd gap (hgb), with adjusted CI
hg = faith[faith['model'] == 'hgb'].sort_values('cutoff_week')
x = np.arange(len(hg)); wd = 0.38
plt.figure(figsize=(6.2, 4))
plt.bar(x - wd/2, hg['raw_gap_imd'], wd, label='raw gap', color='#B4B2A9')
plt.bar(x + wd/2, hg['adj_gap_imd'], wd, label='base-rate adjusted', color='#534AB7',
        yerr=[hg['adj_gap_imd']-hg['adj_lo'], hg['adj_hi']-hg['adj_gap_imd']], capsize=3)
plt.axhline(0, color='#444441', lw=0.8)
plt.xticks(x, [f'wk {int(c)}' for c in hg['cutoff_week']])
plt.ylabel('deprived - affluent deletion-AOPC gap')
plt.title('Faithfulness gap: raw vs base-rate-adjusted (gradient boosting)')
plt.legend(frameon=False); plt.tight_layout()
plt.savefig(FIG / 'fig_faithfulness_raw_vs_adjusted.png', dpi=200); plt.close()
print('saved 3 figures to', FIG)

## What was saved, and how to read it

In `results/`:
* `fairness_summary.csv` - flag/TPR/FPR per group and the demographic-parity, equal-opportunity,
  and FPR gaps with bootstrap CIs, per cutoff x model x protected attribute (C4).
* `faithfulness_adjusted_summary.csv` - predicted-class deletion AOPC (sign-corrected) plus the
  raw and base-rate-adjusted deprived-vs-affluent gaps with CIs.
* `figures/` - equal-opportunity gap by cutoff, corrected AOPC by model, and raw-vs-adjusted gap.

**Reading the headline:** compare `raw_gap_imd` against `adj_gap_imd` with its CI. If the adjusted
gap stays clearly positive and its CI excludes zero, the higher faithfulness for deprived students
is a real effect, not a base-rate artifact, and the C6 claim stands in the corrected direction.
If the adjusted gap collapses toward zero, the raw gap was driven by base risk and should be
reported as such. Either way it is now a defensible, honestly-framed result.

**Next (NB05):** mutability-constrained counterfactuals and recourse-burden disparity (C5),
then the assembled trust-equity table bringing calibration, faithfulness, recourse, and the
fairness gaps together by protected group (C6).